# Santander Cycles Station Analysis using DBSCAN for Atypical Station Detection

## Project Overview

This notebook applies DBSCAN (Density-Based Spatial Clustering of Applications with Noise) to station-level Santander Cycles behaviour data to test whether a density-based clustering approach can reveal meaningful operational structure.

The analysis uses the same engineered station-level feature set developed in the earlier K-Means project. The aim here is not to replace the K-Means work, but to test DBSCAN as an alternative and evaluate whether it is more useful for identifying atypical stations than for full network segmentation.

## Business Problem

Bike-sharing stations do not all behave in the same way. Some stations follow regular commuter-oriented patterns, while others may behave unusually because of demand imbalance, trip timing, or surrounding location dynamics.

From an operational perspective, it is useful not only to segment the main station types, but also to identify stations that do not fit the dominant network pattern. These atypical stations may require targeted monitoring, manual review, or special rebalancing attention.

## Project Objective

The objective of this notebook is to apply DBSCAN to the same station-level behavioural feature set used in the K-Means project and assess whether a density-based clustering method can provide useful additional insight.

In particular, this notebook aims to:
- test DBSCAN as an alternative clustering approach
- evaluate whether DBSCAN can identify meaningful density-based station groups
- determine whether DBSCAN is more useful for atypical station detection than for full operational segmentation

## Why DBSCAN?

Unlike K-Means, DBSCAN does not require the number of clusters to be chosen in advance. Instead, it groups points based on local density and can label isolated observations as noise.

This makes DBSCAN useful when the aim is not only to look for grouped structure, but also to identify stations that do not fit the dominant operational pattern.

## Dataset and Initial Inspection

The dataset used here is a cleaned station-level feature table, where each row represents one station and the columns summarise operational behaviour such as departures, arrivals, trip duration, peak-period activity, and station imbalance.

At this stage, the data is loaded and inspected to confirm its shape, available columns, and overall structure before any clustering is performed.

In [1]:
import pandas as pd

df = pd.read_csv("station_features_cleaned.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

(806, 22)
['station_number', 'station_name', 'total_departures', 'active_days', 'weekday_departures', 'weekend_departures', 'morning_peak_departures', 'evening_peak_departures', 'avg_departure_duration_ms', 'avg_daily_departures', 'weekend_departure_share', 'morning_peak_departure_share', 'evening_peak_departure_share', 'total_arrivals', 'active_days_arr', 'avg_daily_arrivals', 'net_flow', 'imbalance_ratio', 'cluster', 'recommended_action', 'pca1', 'pca2']


,station_number,station_name,total_departures,active_days,weekday_departures,weekend_departures,morning_peak_departures,evening_peak_departures,avg_departure_duration_ms,avg_daily_departures,...,evening_peak_departure_share,total_arrivals,active_days_arr,avg_daily_arrivals,net_flow,imbalance_ratio,cluster,recommended_action,pca1,pca2
0,959.0,"Milroy Walk, South Bank",747.0,30.0,528.0,219.0,80.0,259.0,9.796887e+05,24.900000,...,0.346720,784,30,26.133333,37.0,0.024151,4,Prioritise evening dock relief and destination...,0.412188,-1.277151
1,960.0,"Hop Exchange, The Borough",3552.0,30.0,2385.0,1167.0,335.0,863.0,1.691872e+06,118.400000,...,0.242962,4479,30,149.300000,927.0,0.115413,1,High-priority core station for monitoring and ...,5.380974,2.529871
2,961.0,"Union Street, The Borough",995.0,30.0,721.0,274.0,212.0,259.0,9.294264e+05,33.166667,...,0.260302,1003,30,33.433333,8.0,0.004002,2,Standard commuter-period rebalancing,-0.077756,-0.133806
3,962.0,"Stamford Street, South Bank",726.0,30.0,507.0,219.0,193.0,164.0,1.118847e+06,24.200000,...,0.225895,809,30,26.966667,83.0,0.054036,2,Standard commuter-period rebalancing,-0.961940,0.001791
4,963.0,"Bankside Mix, Bankside",1017.0,30.0,718.0,299.0,113.0,359.0,1.623988e+06,33.900000,...,0.352999,1151,31,37.129032,134.0,0.061780,4,Prioritise evening dock relief and destination...,0.906092,-1.111591


## Preparing the Modelling Data

DBSCAN should be fitted on behavioural features rather than identifiers or previous model outputs.

For this reason, identifier columns such as station number and station name are removed from the modelling input. Previous K-Means outputs and interpretation columns are also removed so that DBSCAN is applied independently on the original behavioural feature space.

In [2]:
import pandas as pd

df = pd.read_csv("station_features_cleaned.csv")

drop_cols = [
    "station_number",
    "station_name",
    "cluster",
    "recommended_action",
    "pca1",
    "pca2"
]

X = df.drop(columns=drop_cols)

print(X.shape)
print(X.columns.tolist())
X.head()

(806, 16)
['total_departures', 'active_days', 'weekday_departures', 'weekend_departures', 'morning_peak_departures', 'evening_peak_departures', 'avg_departure_duration_ms', 'avg_daily_departures', 'weekend_departure_share', 'morning_peak_departure_share', 'evening_peak_departure_share', 'total_arrivals', 'active_days_arr', 'avg_daily_arrivals', 'net_flow', 'imbalance_ratio']


,total_departures,active_days,weekday_departures,weekend_departures,morning_peak_departures,evening_peak_departures,avg_departure_duration_ms,avg_daily_departures,weekend_departure_share,morning_peak_departure_share,evening_peak_departure_share,total_arrivals,active_days_arr,avg_daily_arrivals,net_flow,imbalance_ratio
0,747.0,30.0,528.0,219.0,80.0,259.0,9.796887e+05,24.900000,0.293173,0.107095,0.346720,784,30,26.133333,37.0,0.024151
1,3552.0,30.0,2385.0,1167.0,335.0,863.0,1.691872e+06,118.400000,0.328547,0.094313,0.242962,4479,30,149.300000,927.0,0.115413
2,995.0,30.0,721.0,274.0,212.0,259.0,9.294264e+05,33.166667,0.275377,0.213065,0.260302,1003,30,33.433333,8.0,0.004002
3,726.0,30.0,507.0,219.0,193.0,164.0,1.118847e+06,24.200000,0.301653,0.265840,0.225895,809,30,26.966667,83.0,0.054036
4,1017.0,30.0,718.0,299.0,113.0,359.0,1.623988e+06,33.900000,0.294002,0.111111,0.352999,1151,31,37.129032,134.0,0.061780


## Missing Value Check

Before fitting DBSCAN, the modelling features are checked for missing values. This is important because clustering algorithms require a clean numerical input matrix and should not be run on incomplete data without handling null values first.

In [3]:
print(X.isnull().sum())

total_departures                0
active_days                     0
weekday_departures              0
weekend_departures              0
morning_peak_departures         0
evening_peak_departures         0
avg_departure_duration_ms       0
avg_daily_departures            0
weekend_departure_share         0
morning_peak_departure_share    0
evening_peak_departure_share    0
total_arrivals                  0
active_days_arr                 0
avg_daily_arrivals              0
net_flow                        0
imbalance_ratio                 0
dtype: int64


## Feature Scaling

DBSCAN is a distance-based algorithm, so feature scaling is an essential preprocessing step.

This dataset contains variables on very different numeric ranges, including station counts, average daily flow, ratio-based measures, and trip duration in milliseconds. Without scaling, large-magnitude variables would dominate the distance calculations and distort the clustering result.

To avoid this, the behavioural features are standardised before fitting DBSCAN.

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)

(806, 16)


In [5]:
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled_df.head()

,total_departures,active_days,weekday_departures,weekend_departures,morning_peak_departures,evening_peak_departures,avg_departure_duration_ms,avg_daily_departures,weekend_departure_share,morning_peak_departure_share,evening_peak_departure_share,total_arrivals,active_days_arr,avg_daily_arrivals,net_flow,imbalance_ratio
0,-0.378943,0.065035,-0.348208,-0.378508,-0.582883,-0.022806,-0.432720,-0.384888,-0.277133,-0.807317,1.072208,-0.320262,0.040601,-0.323956,0.348385,-0.085311
1,3.845331,0.065035,3.867345,3.274192,1.063602,2.782317,-0.180887,3.920837,0.190441,-0.944368,-0.150076,5.193392,0.040601,5.297586,8.728445,1.440092
2,-0.005460,0.065035,0.089919,-0.166590,0.269415,-0.022806,-0.450493,-0.004204,-0.512355,0.328924,0.054188,0.006528,0.040601,0.009229,0.075326,-0.422100
3,-0.410568,0.065035,-0.395880,-0.378508,0.146736,-0.464010,-0.383513,-0.417124,-0.165044,0.894790,-0.351120,-0.282958,0.040601,-0.285921,0.781511,0.414204
4,0.027672,0.065035,0.083109,-0.070263,-0.369808,0.441618,-0.204892,0.029567,-0.266172,-0.764255,1.146173,0.227372,0.568409,0.177907,1.261717,0.543627


## Scaled Feature Preview

The scaled feature matrix is converted back into a labelled DataFrame for inspection. This step is not required for DBSCAN itself, but it makes the transformed data easier to read and verify before model fitting.

## Initial DBSCAN Run

A first DBSCAN model is fitted using an initial parameter setting of `eps = 1.2` and `min_samples = 5`.

This first run is used as a diagnostic starting point to see whether DBSCAN identifies several meaningful dense groups or instead treats the dataset as one dominant dense region with some noise points around it.

In [6]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=1.2, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)

df["dbscan_cluster"] = dbscan_labels

print(df["dbscan_cluster"].value_counts().sort_index())

dbscan_cluster
-1    199
 0    607
Name: count, dtype: int64


## Interpreting the First Result

The initial result should not be treated as final. In DBSCAN, the output depends heavily on parameter choice.

If one very large cluster dominates the output, the neighbourhood radius may be too loose. If too many stations are labelled as noise, the model may be too strict. The next step is therefore to test how the clustering changes across different `eps` values.

## Parameter Tuning: Varying `eps`

To explore the sensitivity of the model, DBSCAN is re-run across several `eps` values while keeping `min_samples` fixed.

This helps assess whether a smaller neighbourhood radius can reveal more meaningful density-separated station groups, or whether it simply increases the number of noise-labelled stations.

In [7]:
for eps in [1.0, 0.9, 0.8]:
    dbscan = DBSCAN(eps=eps, min_samples=5)
    labels = dbscan.fit_predict(X_scaled)
    print(f"\neps = {eps}")
    print(pd.Series(labels).value_counts().sort_index())


eps = 1.0
-1    282
 0    510
 1      8
 2      6
Name: count, dtype: int64

eps = 0.9
-1    357
 0    423
 1     11
 2      5
 3      4
 4      6
Name: count, dtype: int64

eps = 0.8
-1    465
 0    323
 1      5
 2      4
 3      5
 4      4
Name: count, dtype: int64


## Interpretation of `eps` Tuning

The `eps` tuning results show that reducing the neighbourhood radius does not produce several strong, well-separated operational groups. Instead, the model continues to produce one dominant cluster together with many noise points and only a few very small edge groups.

This suggests that the station feature space is not strongly organised into multiple clean density-separated clusters under DBSCAN.

## Parameter Tuning: Increasing `min_samples`

A second tuning step is performed by increasing `min_samples` to 10. This makes the density rule stricter, meaning that more local support is required before a station is treated as part of a dense region.

This test helps determine whether DBSCAN can recover a more meaningful structure under a stricter density definition.

In [8]:
for eps in [1.0, 1.1, 1.2]:
    dbscan = DBSCAN(eps=eps, min_samples=10)
    labels = dbscan.fit_predict(X_scaled)
    print(f"\neps = {eps}, min_samples = 10")
    print(pd.Series(labels).value_counts().sort_index())


eps = 1.0, min_samples = 10
-1    372
 0    434
Name: count, dtype: int64

eps = 1.1, min_samples = 10
-1    313
 0    493
Name: count, dtype: int64

eps = 1.2, min_samples = 10
-1    253
 0    553
Name: count, dtype: int64


## Interpretation of the Stricter Density Setting

The stricter setting confirms the same overall pattern: DBSCAN still identifies one dominant dense group and a large number of noise points, rather than several large operationally distinct clusters.

This indicates that DBSCAN is not the strongest method here for full station segmentation, at least on this feature space and time window.

## Inspecting Atypical Stations

Although DBSCAN does not perform strongly here as a full segmentation model, it still provides useful information through its noise labels.

Stations labelled as `-1` do not fit the dominant dense pattern in the dataset. These stations may represent unusual activity profiles, strong imbalance behaviour, special-demand locations, or operational edge cases that deserve separate review.

In [9]:
final_dbscan = DBSCAN(eps=1.2, min_samples=5)
df["dbscan_cluster"] = final_dbscan.fit_predict(X_scaled)

noise_stations = df[df["dbscan_cluster"] == -1].copy()

print("Number of noise stations:", noise_stations.shape[0])

noise_stations[[
    "station_number",
    "station_name",
    "total_departures",
    "total_arrivals",
    "net_flow",
    "imbalance_ratio"
]].head(15)

Number of noise stations: 199


,station_number,station_name,total_departures,total_arrivals,net_flow,imbalance_ratio
1,960.0,"Hop Exchange, The Borough",3552.0,4479,927.0,0.115413
9,968.0,"Warwick Row, Westminster",1673.0,1930,257.0,0.071310
14,973.0,"Bethnal Green Road, Shoreditch",2307.0,2252,-55.0,0.012061
22,982.0,"Holborn Circus, Holborn",1530.0,2102,572.0,0.157446
24,984.0,"Finsbury Circus, Liverpool Street",1870.0,2282,412.0,0.099205
30,991.0,"Crosswall, Tower",1949.0,2150,201.0,0.049024
32,993.0,"Drummond Street , Euston",613.0,507,-106.0,0.094558
33,994.0,"Tanner Street, Bermondsey",1315.0,1381,66.0,0.024472
34,995.0,"Little Argyll Street, West End",2843.0,3015,172.0,0.029357
36,997.0,"Worship Street, Shoreditch",1284.0,1300,16.0,0.006190


## Comparison with the K-Means Baseline

The earlier K-Means analysis was more effective for broad operational segmentation because it produced multiple interpretable station groups across the network.

By contrast, DBSCAN mostly identified one broad dense group and many noise points. This suggests that, for this station feature space, DBSCAN is better used as a supporting method for detecting atypical stations rather than as the main segmentation model.

## Final Conclusion

This notebook applied DBSCAN to the same station-level Santander Cycles feature set used in the earlier K-Means segmentation project.

The main finding is that DBSCAN did not recover several large, meaningful density-based station groups. Instead, it mostly identified one broad dense cluster along with many noise points and only a few very small additional groups.

This means DBSCAN is not the strongest model here for primary station segmentation. However, it still adds value by identifying atypical stations that do not conform to the dominant behavioural pattern.

Taken together, the two analyses provide a stronger combined conclusion:
- K-Means is better suited for broad operational station segmentation
- DBSCAN is more useful for highlighting unusual or non-standard stations for further review

## Limitations

This DBSCAN analysis did not recover several large, well-separated station groups. Across the tested parameter settings, the model mostly produced one dominant cluster together with a large number of noise-labelled stations, which limits its usefulness as a primary segmentation method in this feature space.

The results are also highly sensitive to parameter choice. Small changes in `eps` and `min_samples` changed the number of noise points and small edge clusters noticeably, which means the interpretation is not as stable or straightforward as the K-Means baseline.

Another limitation is that the dataset is based on engineered station-level summary features rather than raw temporal sequences or real-time operational states. This means the analysis captures broad behavioural patterns, but may miss finer-grained short-term variation such as day-specific disruption, event-driven demand surges, or hourly operational instability.

Finally, this notebook is more useful for identifying atypical stations than for producing a full operational station segmentation. In this project, DBSCAN added supporting insight, but it did not outperform the earlier K-Means solution for broad station grouping.